In [3]:
# =============================================================
# 1️⃣ Environment Setup (1985–2024)
# =============================================================
import arcpy, os, glob, math, time
from arcpy.sa import *

# Enable overwrite
arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False  # prevents clutter during batch runs

# Set working directories
arcpy.env.workspace = r"C:\Users\shidebna\Downloads\Thesis_Analysis"
out_dir = os.path.join(arcpy.env.workspace, "WindDirection_Output")
os.makedirs(out_dir, exist_ok=True)

print("✅ ArcPy environment ready")
print("Input Folder :", arcpy.env.workspace)
print("Output Folder:", out_dir)


✅ ArcPy environment ready
Input Folder : C:\Users\shidebna\Downloads\Thesis_Analysis
Output Folder: C:\Users\shidebna\Downloads\Thesis_Analysis\WindDirection_Output


In [4]:
# =============================================================
# 2️⃣ Circular math utilities for wind direction
# =============================================================

def vector_average(raster_list):
    """Compute circular (vector-based) mean of wind direction rasters."""
    n = len(raster_list)
    cos_sum = sum([Cos(Raster(r) * math.pi / 180) for r in raster_list])
    sin_sum = sum([Sin(Raster(r) * math.pi / 180) for r in raster_list])
    cos_mean = cos_sum / n
    sin_mean = sin_sum / n
    mean_dir = ATan2(sin_mean, cos_mean) * 180 / math.pi
    mean_dir = (mean_dir + 360) % 360
    return mean_dir

def angular_diff(r1, r2):
    """Return ±180° normalized angular difference between two rasters."""
    raw = Raster(r2) - Raster(r1)
    adj = Con(raw > 180, raw - 360,
              Con(raw < -180, raw + 360, raw))
    return adj

print("✅ Circular averaging and difference functions loaded")


✅ Circular averaging and difference functions loaded


In [5]:
# =============================================================
# 3️⃣ Define meteorological seasons & file lookup
# =============================================================
seasons = {
    "DJF": [12, 1, 2],
    "MAM": [3, 4, 5],
    "JJA": [6, 7, 8],
    "SON": [9, 10, 11]
}

def find_monthly_files(year, months):
    paths = []
    for m in months:
        y = year - 1 if m == 12 else year
        pattern = f"RWD_Global_{y}_{m}.tif"
        paths.extend(glob.glob(os.path.join(arcpy.env.workspace, pattern)))
    return sorted(paths)

print("✅ Season dictionary and file finder ready")


✅ Season dictionary and file finder ready


In [6]:
# =============================================================
# 4️⃣  Seasonal Mean Computation (DJF, MAM, JJA, SON: 1984–2024)
#     → All outputs go to:
#       C:\Users\shidebna\Downloads\Thesis_Analysis\WindDirection_Output\seasonal_mean
# =============================================================

import os, glob, time

# ✅ Create a dedicated output directory
main_out = r"C:\Users\shidebna\Downloads\Thesis_Analysis\WindDirection_Output"
out_dir = os.path.join(main_out, "seasonal_mean")
os.makedirs(out_dir, exist_ok=True)

print("✅ Output directory confirmed:", out_dir)


# =============================================================
# Helper: Compute mean for a given list of months
# =============================================================
def compute_custom_mean(month_list, out_name):
    """Compute circular (vector-based) mean for a custom set of months."""
    start = time.time()
    all_files = []

    for (y, m) in month_list:
        # Support both padded & non-padded month naming
        pattern1 = f"RWD_Global_{y}_{m}.tif"
        pattern2 = f"RWD_Global_{y}_{m:02d}.tif"
        found = glob.glob(os.path.join(arcpy.env.workspace, pattern1))
        if not found:
            found = glob.glob(os.path.join(arcpy.env.workspace, pattern2))
        all_files.extend(found)
        print(f"📂 {y}-{m:02d}: {len(found)} file(s) found", flush=True)

    if not all_files:
        print(f"⚠️ No rasters found for {out_name}, skipped.")
        return

    print(f"⏳ Computing mean for {len(all_files)} rasters ({out_name}) ...", flush=True)
    mean_ras = vector_average(all_files)

    # Save to seasonal_mean folder
    out_path = os.path.join(out_dir, f"{out_name}.tif")
    mean_ras.save(out_path)

    print(f"✅ Saved {os.path.basename(out_path)} in {(time.time()-start):.1f}s", flush=True)


# =============================================================
# Main: Compute all 4 seasonal means per year (1984–2024)
# =============================================================
def compute_all_season_means(start_year=1984, end_year=2024):
    """
    Automatically compute circular means for all four meteorological seasons:
      DJF_1985 = Dec_1984 + Jan_1985 + Feb_1985
      MAM_1985 = Mar_1985 + Apr_1985 + May_1985
      JJA_1985 = Jun_1985 + Jul_1985 + Aug_1985
      SON_1985 = Sep_1985 + Oct_1985 + Nov_1985
      ...
      until 2024
    """
    total_start = time.time()
    log_file = os.path.join(out_dir, "seasonal_auto_processing_log.txt")

    with open(log_file, "a", encoding="utf-8") as log:
        log.write(f"\n========== Seasonal Auto Run: {time.ctime()} ==========\n")

    # Loop through each year
    for y in range(start_year + 1, end_year + 1):
        print(f"\n🕒 YEAR {y} — processing all seasons", flush=True)
        with open(log_file, "a", encoding="utf-8") as log:
            log.write(f"\n🕒 YEAR {y}\n")

        # Define seasons for the current year
        seasons = {
            "DJF": [(y - 1, 12), (y, 1), (y, 2)],
            "MAM": [(y, 3), (y, 4), (y, 5)],
            "JJA": [(y, 6), (y, 7), (y, 8)],
            "SON": [(y, 9), (y, 10), (y, 11)]
        }

        # Compute each season
        for s, months in seasons.items():
            out_name = f"{s}Mean_{y}"
            print(f"➡️  {s} {y}: months {[m for (_, m) in months]}", flush=True)
            try:
                compute_custom_mean(months, out_name)
                with open(log_file, "a", encoding="utf-8") as log:
                    log.write(f"✅ {out_name} done\n")
            except Exception as e:
                err = f"❌ {s} {y} failed — {e}"
                print(err, flush=True)
                with open(log_file, "a", encoding="utf-8") as log:
                    log.write(err + "\n")

    total_msg = f"\n🎯 All seasonal means complete in {(time.time()-total_start)/60:.2f} min\n"
    print(total_msg)
    with open(log_file, "a", encoding="utf-8") as log:
        log.write(total_msg)


# ▶️ TEST FIRST (short span)
compute_all_season_means(1984, 2024)

# ▶️ FULL RUN
# compute_all_season_means(1984, 2024)


✅ Output directory confirmed: C:\Users\shidebna\Downloads\Thesis_Analysis\WindDirection_Output\seasonal_mean

🕒 YEAR 1985 — processing all seasons
➡️  DJF 1985: months [12, 1, 2]
📂 1984-12: 0 file(s) found
📂 1985-01: 0 file(s) found
📂 1985-02: 0 file(s) found
⚠️ No rasters found for DJFMean_1985, skipped.
➡️  MAM 1985: months [3, 4, 5]
📂 1985-03: 0 file(s) found
📂 1985-04: 0 file(s) found
📂 1985-05: 0 file(s) found
⚠️ No rasters found for MAMMean_1985, skipped.
➡️  JJA 1985: months [6, 7, 8]
📂 1985-06: 0 file(s) found
📂 1985-07: 0 file(s) found
📂 1985-08: 0 file(s) found
⚠️ No rasters found for JJAMean_1985, skipped.
➡️  SON 1985: months [9, 10, 11]
📂 1985-09: 0 file(s) found
📂 1985-10: 0 file(s) found
📂 1985-11: 0 file(s) found
⚠️ No rasters found for SONMean_1985, skipped.

🕒 YEAR 1986 — processing all seasons
➡️  DJF 1986: months [12, 1, 2]
📂 1985-12: 0 file(s) found
📂 1986-01: 0 file(s) found
📂 1986-02: 0 file(s) found
⚠️ No rasters found for DJFMean_1986, skipped.
➡️  MAM 1986: mo

In [14]:
# =============================================================
# 5️⃣ Combine seasonal means into 10-year composites (1985–2024)
# =============================================================

import os, glob, time

# ✅ Create a subfolder for 10-year means
main_out = r"C:\Users\shidebna\Downloads\Thesis_Analysis\WindDirection_Output"
season_dir = os.path.join(main_out, "seasonal_mean")
ten_year_dir = os.path.join(main_out, "ten_year_mean")
os.makedirs(ten_year_dir, exist_ok=True)

print("✅ Output directory confirmed:", ten_year_dir)


def compute_ten_year_means(start=1985, end=2024):
    """
    Compute 10-year composite means for each season (DJF, MAM, JJA, SON)
    using circular vector averaging across the seasonal mean rasters.
    """
    total_start = time.time()
    log_file = os.path.join(ten_year_dir, "ten_year_processing_log.txt")

    with open(log_file, "a", encoding="utf-8") as log:
        log.write(f"\n========== 10-Year Composite Run: {time.ctime()} ==========\n")

    # Define 10-year blocks
    blocks = list(range(start, end + 1, 10))
    if blocks[-1] != end:
        blocks.append(end)

    # Seasons list
    seasons = ["DJF", "MAM", "JJA", "SON"]

    # Loop over 10-year blocks
    for y in range(start, end, 10):
        y2 = min(y + 9, end)
        print(f"\n🕒 10-Year Block {y}-{y2}", flush=True)
        with open(log_file, "a", encoding="utf-8") as log:
            log.write(f"\n🕒 10-Year Block {y}-{y2}\n")

        # Process each season
        for s in seasons:
            # Gather all matching seasonal mean rasters
            pattern = os.path.join(season_dir, f"{s}Mean_*.tif")
            all_files = [f for f in glob.glob(pattern)
                         if y <= int(os.path.basename(f).split("_")[-1].split(".")[0]) <= y2]

            if not all_files:
                print(f"⚠️  No {s} rasters found for {y}-{y2}, skipped.", flush=True)
                with open(log_file, "a", encoding="utf-8") as log:
                    log.write(f"⚠️  No {s} rasters found for {y}-{y2}\n")
                continue

            # Compute circular mean
            t0 = time.time()
            print(f"⏳  Computing {s} {y}-{y2} from {len(all_files)} rasters...", flush=True)
            mean10 = vector_average(all_files)
            out = os.path.join(ten_year_dir, f"{s}_10YearMean_{y}-{y2}.tif")
            mean10.save(out)

            done = f"✅  Saved {os.path.basename(out)} ({(time.time()-t0):.1f}s)"
            print(done, flush=True)
            with open(log_file, "a", encoding="utf-8") as log:
                log.write(done + "\n")

    total_msg = f"\n🎯 All 10-year composites complete in {(time.time()-total_start)/60:.2f} min\n"
    print(total_msg)
    with open(log_file, "a", encoding="utf-8") as log:
        log.write(total_msg)


# ▶️ TEST FIRST (short block)
compute_ten_year_means(1985, 2024)

# ▶️ FULL RUN
# compute_ten_year_means(1985, 2024)


✅ Output directory confirmed: C:\Users\shidebna\Downloads\Thesis_Analysis\WindDirection_Output\ten_year_mean

🕒 10-Year Block 1985-1994
⏳  Computing DJF 1985-1994 from 10 rasters...
✅  Saved DJF_10YearMean_1985-1994.tif (2.0s)
⏳  Computing MAM 1985-1994 from 10 rasters...
✅  Saved MAM_10YearMean_1985-1994.tif (1.8s)
⏳  Computing JJA 1985-1994 from 10 rasters...
✅  Saved JJA_10YearMean_1985-1994.tif (1.8s)
⏳  Computing SON 1985-1994 from 10 rasters...
✅  Saved SON_10YearMean_1985-1994.tif (1.8s)

🕒 10-Year Block 1995-2004
⏳  Computing DJF 1995-2004 from 10 rasters...
✅  Saved DJF_10YearMean_1995-2004.tif (1.9s)
⏳  Computing MAM 1995-2004 from 10 rasters...
✅  Saved MAM_10YearMean_1995-2004.tif (1.8s)
⏳  Computing JJA 1995-2004 from 10 rasters...
✅  Saved JJA_10YearMean_1995-2004.tif (1.9s)
⏳  Computing SON 1995-2004 from 10 rasters...
✅  Saved SON_10YearMean_1995-2004.tif (1.9s)

🕒 10-Year Block 2005-2014
⏳  Computing DJF 2005-2014 from 10 rasters...
✅  Saved DJF_10YearMean_2005-2014.ti

In [18]:
# =============================================================
# 5b️⃣ Compute Decadal Averages (1985–2024) not seasoal yearl average 
# =============================================================

import os
import time
from arcpy.sa import Raster

# ✅ Create a subfolder for decadal means
decadal_mean_dir = os.path.join(main_out, "New decadal_mean")
os.makedirs(decadal_mean_dir, exist_ok=True)

print("✅ Output directory for decadal means confirmed:", decadal_mean_dir)

def compute_decadal_mean(start=1985, end=2024):
    """
    Compute decadal mean by combining DJF, MAM, JJA, and SON seasonal mean rasters
    into a decadal average for each decade (e.g., 1985-1994, 1995-2004, etc.).
    """
    total_start = time.time()
    log_file = os.path.join(decadal_mean_dir, "decadal_processing_log.txt")

    with open(log_file, "a", encoding="utf-8") as log:
        log.write(f"\n========== Decadal Mean Run: {time.ctime()} ==========\n")

    # Define the season directory (seasonal mean rasters)
    season_dir = os.path.join(main_out, "ten_year_mean")  # Use the correct directory for the seasonal means

    # Define seasons
    seasons = ["DJF", "MAM", "JJA", "SON"]

    # Loop over each decade in the period
    for y in range(start, end, 10):
        y2 = min(y + 9, end)
        print(f"\n🕒 Decade {y}-{y2} — processing decadal mean", flush=True)
        with open(log_file, "a", encoding="utf-8") as log:
            log.write(f"\n🕒 Decade {y}-{y2}\n")

        seasonal_rasters = []

        # Gather seasonal rasters for the given decade
        for season in seasons:
            raster_path = os.path.join(season_dir, f"{season}_10YearMean_{y}-{y2}.tif")
            if arcpy.Exists(raster_path):
                seasonal_rasters.append(Raster(raster_path))
            else:
                print(f"⚠️  Missing raster for {season} {y}-{y2}, skipped.")
        
        if len(seasonal_rasters) == len(seasons):  # Ensure all seasons are present
            # Compute the average for the decade (sum of all seasons divided by the number of seasons)
            decadal_mean = sum(seasonal_rasters) / len(seasonal_rasters)

            # Set the output file path for decadal mean raster
            out_path = os.path.join(decadal_mean_dir, f"DecadalMean_{y}-{y2}.tif")

            # Debugging print to check the output path
            print(f"Saving output to: {out_path}")  # This will print the path to the console

            # Save the decadal mean raster
            decadal_mean.save(out_path)
            print(f"✅ Saved {os.path.basename(out_path)}")
        else:
            print(f"⚠️  Could not compute decadal mean for {y}-{y2}, missing one or more seasons.")

    total_msg = f"\n🎯 All decadal means complete in {(time.time() - total_start) / 60:.2f} min\n"
    print(total_msg)
    with open(log_file, "a", encoding="utf-8") as log:
        log.write(total_msg)

# ▶️ RUN
compute_decadal_mean(1985, 2024)

✅ Output directory for decadal means confirmed: C:\Users\shidebna\Downloads\Thesis_Analysis\WindDirection_Output\New decadal_mean

🕒 Decade 1985-1994 — processing decadal mean
Saving output to: C:\Users\shidebna\Downloads\Thesis_Analysis\WindDirection_Output\New decadal_mean\DecadalMean_1985-1994.tif
✅ Saved DecadalMean_1985-1994.tif

🕒 Decade 1995-2004 — processing decadal mean
Saving output to: C:\Users\shidebna\Downloads\Thesis_Analysis\WindDirection_Output\New decadal_mean\DecadalMean_1995-2004.tif
✅ Saved DecadalMean_1995-2004.tif

🕒 Decade 2005-2014 — processing decadal mean
Saving output to: C:\Users\shidebna\Downloads\Thesis_Analysis\WindDirection_Output\New decadal_mean\DecadalMean_2005-2014.tif
✅ Saved DecadalMean_2005-2014.tif

🕒 Decade 2015-2024 — processing decadal mean
Saving output to: C:\Users\shidebna\Downloads\Thesis_Analysis\WindDirection_Output\New decadal_mean\DecadalMean_2015-2024.tif
✅ Saved DecadalMean_2015-2024.tif

🎯 All decadal means complete in 0.01 min



In [34]:
# =============================================================
# 6️⃣ ±180° Directional Change Between 10-Year Intervals
# =============================================================
import os, time

def compute_directional_change(season, early_decade, late_decade):
    """Compute pixel-by-pixel angular change between two decade means."""
    start = time.time()
    r1 = os.path.join(out_dir_10yr, f"{season}_10YearMean_{early_decade}.tif")
    r2 = os.path.join(out_dir_10yr, f"{season}_10YearMean_{late_decade}.tif")
    if not (arcpy.Exists(r1) and arcpy.Exists(r2)):
        print(f"⚠️ Missing rasters for {season} {early_decade} or {late_decade}")
        return

    print(f"⏳ Calculating {season} change: {early_decade} → {late_decade}", flush=True)
    diff = angular_diff(r1, r2)
    out = os.path.join(out_dir_change, f"{season}_Change_{early_decade}_vs_{late_decade}.tif")
    diff.save(out)
    print(f"✅ Saved {os.path.basename(out)} ({(time.time()-start):.1f}s)", flush=True)


def compute_all_directional_changes():
    """Run all decadal and long-term directional changes for each season."""
    total_start = time.time()
    decades = ["1985-1994", "1995-2004", "2005-2014", "2015-2024"]
    seasons = ["DJF", "MAM", "JJA", "SON"]

    print("🌎 Starting decadal directional-change analysis...\n")

    # Pair consecutive decades
    for s in seasons:
        for i in range(len(decades) - 1):
            d1, d2 = decades[i], decades[i + 1]
            compute_directional_change(s, d1, d2)

        # Earliest → Latest long-term change
        compute_directional_change(s, decades[0], decades[-1])

    print(f"\n🎯 All directional-change rasters complete in {(time.time()-total_start)/60:.2f} min\n")


# =============================================================
# 🗂️ Prepare output folder structure
# =============================================================
base_dir = r"C:\Users\shidebna\Downloads\Thesis_Analysis\WindDirection_Output"
out_dir_10yr = os.path.join(base_dir, "ten_year_mean")
out_dir_change = os.path.join(base_dir, "decadal_change")
os.makedirs(out_dir_change, exist_ok=True)

# ▶️ RUN:
compute_all_directional_changes()


🌎 Starting decadal directional-change analysis...

⏳ Calculating DJF change: 1985-1994 → 1995-2004
✅ Saved DJF_Change_1985-1994_vs_1995-2004.tif (0.3s)
⏳ Calculating DJF change: 1995-2004 → 2005-2014
✅ Saved DJF_Change_1995-2004_vs_2005-2014.tif (0.3s)
⏳ Calculating DJF change: 2005-2014 → 2015-2024
✅ Saved DJF_Change_2005-2014_vs_2015-2024.tif (0.3s)
⏳ Calculating DJF change: 1985-1994 → 2015-2024
✅ Saved DJF_Change_1985-1994_vs_2015-2024.tif (0.3s)
⏳ Calculating MAM change: 1985-1994 → 1995-2004
✅ Saved MAM_Change_1985-1994_vs_1995-2004.tif (0.3s)
⏳ Calculating MAM change: 1995-2004 → 2005-2014
✅ Saved MAM_Change_1995-2004_vs_2005-2014.tif (0.3s)
⏳ Calculating MAM change: 2005-2014 → 2015-2024
✅ Saved MAM_Change_2005-2014_vs_2015-2024.tif (0.3s)
⏳ Calculating MAM change: 1985-1994 → 2015-2024
✅ Saved MAM_Change_1985-1994_vs_2015-2024.tif (0.3s)
⏳ Calculating JJA change: 1985-1994 → 1995-2004
✅ Saved JJA_Change_1985-1994_vs_1995-2004.tif (0.3s)
⏳ Calculating JJA change: 1995-2004 → 20

In [19]:
# =============================================================
# 6️⃣.B Compute Decadal Change (1985–2024) Not seasonal (year averade)
# =============================================================


import os
import time
from arcpy.sa import Raster, Con

# ✅ Create a subfolder for decadal change rasters
new_decadal_change_dir = os.path.join(main_out, "new_decadal_change")  # New output folder
os.makedirs(new_decadal_change_dir, exist_ok=True)

print("✅ New output directory for decadal change confirmed:", new_decadal_change_dir)

def angular_diff(r1, r2):
    """Compute ±180° angular difference between two rasters."""
    # Compute the raw difference between the two rasters
    raw = r2 - r1
    
    # Normalize the difference to the range ±180°
    adj = Con(raw > 180, raw - 360, 
              Con(raw < -180, raw + 360, raw))
    return adj

def compute_decadal_change(start=1985, end=2024):
    """
    Compute decadal change by comparing the decadal means for each pair of consecutive decades and non-consecutive decades.
    """
    total_start = time.time()
    log_file = os.path.join(new_decadal_change_dir, "decadal_change_log.txt")  # Log file in new folder

    with open(log_file, "a", encoding="utf-8") as log:
        log.write(f"\n========== Decadal Change Run: {time.ctime()} ==========\n")

    # Loop over each decade in the period
    decades = [f"{start}-{start+9}", f"{start+10}-{start+19}", f"{start+20}-{start+29}", f"{start+30}-{start+39}"]
    
    # Add additional non-consecutive decade comparison (1985-1994 vs 2015-2024)
    extra_comparison = [(f"{start}-{start+9}", f"{start+30}-{start+39}")]

    # Combine consecutive decade comparisons with the extra comparison
    all_comparisons = [(decades[i], decades[i + 1]) for i in range(len(decades) - 1)] + extra_comparison

    # Loop through each decade pair and compute change
    for decade1, decade2 in all_comparisons:
        
        print(f"\n🕒 Comparing {decade1} vs {decade2} — processing decadal change", flush=True)
        with open(log_file, "a", encoding="utf-8") as log:
            log.write(f"\n🕒 Comparing {decade1} vs {decade2}\n")

        # Load the decadal mean rasters for the two decades
        r1_path = os.path.join(decadal_mean_dir, f"DecadalMean_{decade1}.tif")
        r2_path = os.path.join(decadal_mean_dir, f"DecadalMean_{decade2}.tif")
        
        if arcpy.Exists(r1_path) and arcpy.Exists(r2_path):
            r1 = Raster(r1_path)
            r2 = Raster(r2_path)

            # Compute the angular difference between the two rasters (±180° normalization)
            decadal_change_raster = angular_diff(r1, r2)

            # Save the decadal change raster in the new output folder
            out_path = os.path.join(new_decadal_change_dir, f"DecadalChange_{decade1}_vs_{decade2}.tif")
            print(f"Saving output to: {out_path}")  # Debugging print to check the output path

            decadal_change_raster.save(out_path)
            print(f"✅ Saved {os.path.basename(out_path)}")
        else:
            print(f"⚠️  Missing rasters for {decade1} or {decade2}, skipped.")

    total_msg = f"\n🎯 All decadal changes complete in {(time.time() - total_start) / 60:.2f} min\n"
    print(total_msg)
    with open(log_file, "a", encoding="utf-8") as log:
        log.write(total_msg)

# ▶️ RUN
compute_decadal_change(1985, 2024)

✅ New output directory for decadal change confirmed: C:\Users\shidebna\Downloads\Thesis_Analysis\WindDirection_Output\new_decadal_change

🕒 Comparing 1985-1994 vs 1995-2004 — processing decadal change
Saving output to: C:\Users\shidebna\Downloads\Thesis_Analysis\WindDirection_Output\new_decadal_change\DecadalChange_1985-1994_vs_1995-2004.tif
✅ Saved DecadalChange_1985-1994_vs_1995-2004.tif

🕒 Comparing 1995-2004 vs 2005-2014 — processing decadal change
Saving output to: C:\Users\shidebna\Downloads\Thesis_Analysis\WindDirection_Output\new_decadal_change\DecadalChange_1995-2004_vs_2005-2014.tif
✅ Saved DecadalChange_1995-2004_vs_2005-2014.tif

🕒 Comparing 2005-2014 vs 2015-2024 — processing decadal change
Saving output to: C:\Users\shidebna\Downloads\Thesis_Analysis\WindDirection_Output\new_decadal_change\DecadalChange_2005-2014_vs_2015-2024.tif
✅ Saved DecadalChange_2005-2014_vs_2015-2024.tif

🕒 Comparing 1985-1994 vs 2015-2024 — processing decadal change
Saving output to: C:\Users\shid

In [17]:
# =============================================================
# 6️⃣C Multi-Interval Directional Changes (10-, 20-, 30-, 40-year)
# =============================================================
import os, time

base_dir = r"C:\Users\shidebna\Downloads\Thesis_Analysis\WindDirection_Output"
out_dir_10yr = os.path.join(base_dir, "ten_year_mean")
out_dir_change = os.path.join(base_dir, "multi_interval_change")
os.makedirs(out_dir_change, exist_ok=True)

def compute_directional_change(season, early_decade, late_decade):
    """Compute ±180° angular difference between two decadal mean rasters."""
    r1 = os.path.join(out_dir_10yr, f"{season}_10YearMean_{early_decade}.tif")
    r2 = os.path.join(out_dir_10yr, f"{season}_10YearMean_{late_decade}.tif")
    if not (arcpy.Exists(r1) and arcpy.Exists(r2)):
        print(f"⚠️ Missing rasters for {season} {early_decade} or {late_decade}")
        return
    print(f"⏳ {season}: {early_decade} → {late_decade}")
    diff = angular_diff(r1, r2)
    out_name = f"{season}_Change_{early_decade}_vs_{late_decade}.tif"
    diff.save(os.path.join(out_dir_change, out_name))
    print(f"✅ Saved {out_name}")

def compute_multi_interval_changes():
    """Compute 10-, 20-, 30-, and 40-year directional changes for all seasons."""
    seasons = ["DJF", "MAM", "JJA", "SON"]
    decades = ["1985-1994", "1995-2004", "2005-2014", "2015-2024"]
    offsets = [1, 2, 3]  # decade gaps → 10, 20, 30 years

    total_start = time.time()
    print("🌎 Starting multi-interval directional-change analysis...\n")

    for s in seasons:
        for i in range(len(decades)):
            for off in offsets:
                j = i + off
                if j >= len(decades):
                    continue
                d1, d2 = decades[i], decades[j]
                compute_directional_change(s, d1, d2)
        # full-period 40-year change
        compute_directional_change(s, decades[0], decades[-1])

    print(f"\n🎯 All multi-interval changes complete in {(time.time()-total_start)/60:.2f} min\n")

# ▶️ RUN
compute_multi_interval_changes()


🌎 Starting multi-interval directional-change analysis...

⏳ DJF: 1985-1994 → 1995-2004
✅ Saved DJF_Change_1985-1994_vs_1995-2004.tif
⏳ DJF: 1985-1994 → 2005-2014
✅ Saved DJF_Change_1985-1994_vs_2005-2014.tif
⏳ DJF: 1985-1994 → 2015-2024
✅ Saved DJF_Change_1985-1994_vs_2015-2024.tif
⏳ DJF: 1995-2004 → 2005-2014
✅ Saved DJF_Change_1995-2004_vs_2005-2014.tif
⏳ DJF: 1995-2004 → 2015-2024
✅ Saved DJF_Change_1995-2004_vs_2015-2024.tif
⏳ DJF: 2005-2014 → 2015-2024
✅ Saved DJF_Change_2005-2014_vs_2015-2024.tif
⏳ DJF: 1985-1994 → 2015-2024
✅ Saved DJF_Change_1985-1994_vs_2015-2024.tif
⏳ MAM: 1985-1994 → 1995-2004
✅ Saved MAM_Change_1985-1994_vs_1995-2004.tif
⏳ MAM: 1985-1994 → 2005-2014
✅ Saved MAM_Change_1985-1994_vs_2005-2014.tif
⏳ MAM: 1985-1994 → 2015-2024
✅ Saved MAM_Change_1985-1994_vs_2015-2024.tif
⏳ MAM: 1995-2004 → 2005-2014
✅ Saved MAM_Change_1995-2004_vs_2005-2014.tif
⏳ MAM: 1995-2004 → 2015-2024
✅ Saved MAM_Change_1995-2004_vs_2015-2024.tif
⏳ MAM: 2005-2014 → 2015-2024
✅ Saved MAM_C

In [18]:
# =============================================================
# 7️⃣  Regional Summaries (Final, Case-Insensitive + Robust)
# =============================================================
import arcpy, os, time

# --- Folder setup ---
base_dir = r"C:\Users\shidebna\Downloads\Thesis_Analysis\WindDirection_Output"
zones = os.path.join(base_dir, "LatBands.shp")
out_dir_zonal = os.path.join(base_dir, "zonal_summary")
change_dir = os.path.join(base_dir, "decadal_change")
os.makedirs(out_dir_zonal, exist_ok=True)

# =============================================================
# 🌎 Step 1 — Ensure Latitude-Band Shapefile Exists
# =============================================================
if not arcpy.Exists(zones):
    print("⚙️  Creating latitude-band shapefile ...")
    arcpy.management.CreateFishnet(
        out_feature_class=zones,
        origin_coord="-180 -90",
        y_axis_coord="-180 -80",
        cell_width="360",      # full globe
        cell_height="10",      # 10° bands
        number_rows="18",      # −90→+90
        number_columns="1",
        geometry_type="POLYGON"
    )

# --- Check existing fields (case-insensitive) ---
existing = [f.name.lower() for f in arcpy.ListFields(zones)]
if "id" not in existing:
    arcpy.management.AddField(zones, "ID", "SHORT")
if "latcenter" not in existing:
    arcpy.management.AddField(zones, "LatCenter", "DOUBLE")

# --- Populate IDs & latitude centers ---
with arcpy.da.UpdateCursor(zones, ["ID", "LatCenter", "SHAPE@"]) as cur:
    i = 1
    for row in cur:
        ext = row[2].extent
        row[0] = i
        row[1] = (ext.YMin + ext.YMax) / 2.0
        cur.updateRow(row)
        i += 1

print(f"✅ Latitude bands ready → {zones}")

# =============================================================
# 📊 Step 2 — Zonal Statistics for One Raster
# =============================================================
def zonal_summary(change_raster, out_table):
    start = time.time()
    arcpy.sa.ZonalStatisticsAsTable(zones, "ID", change_raster, out_table, "DATA", "MEAN")
    print(f"✅ {os.path.basename(out_table)}  ({(time.time()-start):.1f}s)")

# =============================================================
# 🌀 Step 3 — Run for All Change Rasters
# =============================================================
def compute_all_zonal_summaries():
    total_start = time.time()
    if not os.path.exists(change_dir):
        print("⚠️  decadal_change folder not found.")
        return

    tifs = [f for f in os.listdir(change_dir) if f.lower().endswith(".tif")]
    if not tifs:
        print("⚠️  No .tif files found in decadal_change folder.")
        return

    print(f"🔍 Found {len(tifs)} change rasters to process.\n")

    for tif in tifs:
        in_raster = os.path.join(change_dir, tif)
        out_table = os.path.join(out_dir_zonal, os.path.splitext(tif)[0] + ".dbf")
        try:
            zonal_summary(in_raster, out_table)
        except Exception as e:
            print(f"❌  {tif} failed → {e}")

    print(f"\n🎯 All zonal summaries complete in {(time.time()-total_start)/60:.2f} min")

# ▶️ RUN
compute_all_zonal_summaries()


✅ Latitude bands ready → C:\Users\shidebna\Downloads\Thesis_Analysis\WindDirection_Output\LatBands.shp
🔍 Found 16 change rasters to process.

✅ DJF_Change_1985-1994_vs_1995-2004.dbf  (1.0s)
✅ DJF_Change_1985-1994_vs_2015-2024.dbf  (0.4s)
✅ DJF_Change_1995-2004_vs_2005-2014.dbf  (0.4s)
✅ DJF_Change_2005-2014_vs_2015-2024.dbf  (0.5s)
✅ JJA_Change_1985-1994_vs_1995-2004.dbf  (0.6s)
✅ JJA_Change_1985-1994_vs_2015-2024.dbf  (0.4s)
✅ JJA_Change_1995-2004_vs_2005-2014.dbf  (0.5s)
✅ JJA_Change_2005-2014_vs_2015-2024.dbf  (0.5s)
✅ MAM_Change_1985-1994_vs_1995-2004.dbf  (0.5s)
✅ MAM_Change_1985-1994_vs_2015-2024.dbf  (0.5s)
✅ MAM_Change_1995-2004_vs_2005-2014.dbf  (0.5s)
✅ MAM_Change_2005-2014_vs_2015-2024.dbf  (0.4s)
✅ SON_Change_1985-1994_vs_1995-2004.dbf  (0.5s)
✅ SON_Change_1985-1994_vs_2015-2024.dbf  (0.4s)
✅ SON_Change_1995-2004_vs_2005-2014.dbf  (0.5s)
✅ SON_Change_2005-2014_vs_2015-2024.dbf  (0.4s)

🎯 All zonal summaries complete in 0.13 min


In [19]:
# =============================================================
# 8️⃣ Summary of Outputs (Final Research Version)
# =============================================================
import os, glob, pandas as pd

base_dir = r"C:\Users\shidebna\Downloads\Thesis_Analysis\WindDirection_Output"
folders = [
    ("Seasonal Means", os.path.join(base_dir, "seasonal_mean")),
    ("Ten-Year Means", os.path.join(base_dir, "ten_year_mean")),
    ("Decadal Changes", os.path.join(base_dir, "decadal_change")),
    ("Zonal Summaries", os.path.join(base_dir, "zonal_summary")),
]

summary_records = []

print("\n📁 ===== FINAL OUTPUT SUMMARY =====\n")
for label, path in folders:
    if not os.path.exists(path):
        print(f"⚠️  {label}: folder not found ({path})")
        continue
    files = sorted(glob.glob(os.path.join(path, "*.*")))
    print(f"📂 {label}: {len(files)} files")
    for f in files:
        fname = os.path.basename(f)
        print("   •", fname)
        summary_records.append({"Category": label, "File": fname, "Path": f})

# Save summary as CSV (for documentation)
summary_csv = os.path.join(base_dir, "Output_File_Summary.csv")
pd.DataFrame(summary_records).to_csv(summary_csv, index=False)
print(f"\n✅ Output summary saved to {summary_csv}")



📁 ===== FINAL OUTPUT SUMMARY =====

📂 Seasonal Means: 472 files
   • DJFMean_1985.tfw
   • DJFMean_1985.tif
   • DJFMean_1985.tif.aux.xml
   • DJFMean_1986.tfw
   • DJFMean_1986.tif
   • DJFMean_1986.tif.aux.xml
   • DJFMean_1987.tfw
   • DJFMean_1987.tif
   • DJFMean_1987.tif.aux.xml
   • DJFMean_1988.tfw
   • DJFMean_1988.tif
   • DJFMean_1988.tif.aux.xml
   • DJFMean_1989.tfw
   • DJFMean_1989.tif
   • DJFMean_1989.tif.aux.xml
   • DJFMean_1990.tfw
   • DJFMean_1990.tif
   • DJFMean_1990.tif.aux.xml
   • DJFMean_1991.tfw
   • DJFMean_1991.tif
   • DJFMean_1991.tif.aux.xml
   • DJFMean_1992.tfw
   • DJFMean_1992.tif
   • DJFMean_1992.tif.aux.xml
   • DJFMean_1993.tfw
   • DJFMean_1993.tif
   • DJFMean_1993.tif.aux.xml
   • DJFMean_1994.tfw
   • DJFMean_1994.tif
   • DJFMean_1994.tif.aux.xml
   • DJFMean_1995.tfw
   • DJFMean_1995.tif
   • DJFMean_1995.tif.aux.xml
   • DJFMean_1996.tfw
   • DJFMean_1996.tif
   • DJFMean_1996.tif.aux.xml
   • DJFMean_1997.tfw
   • DJFMean_1997.tif
   